# 5B · Cohorts, Drivers & the Traps of "Why"
### Financial Analytics — Module 5

Question: **"why are clients leaving?"** This notebook is the honest toolkit for answering it — and, just as importantly, the gallery of ways the answer goes wrong:

1. **Cohort analysis** — follow groups that started together
2. **Driver hunting** — group comparisons, done with discipline
3. **The mix trap** — adjust before you blame (the RM league table)
4. **The hidden-populations trap** — when the average conceals opposite truths
5. **Correlation ≠ causation** — what diagnostics can and cannot claim

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

BASE = "data/"
cli = pd.read_csv(BASE + "client_book.csv", parse_dates=["onboard_date"])
print(cli.shape, "| overall churn:", round(cli["churned"].mean(), 3))

---
## 1. Cohorts: follow the people who started together

A **cohort** is a group defined by a shared starting moment — everyone onboarded in 2022, everyone who got a loan in Q3. Comparing cohorts separates *"our clients are changing"* from *"our clients age predictably."* 

In [ ]:
cli["cohort"] = cli["onboard_date"].dt.year

cohorts = cli.groupby("cohort").agg(
    clients=("client_id", "count"),
    churn_rate=("churned", "mean"),
    avg_tenure=("tenure_months", "mean"),
).round(3)
print(cohorts)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(cohorts.index.astype(str), cohorts["churn_rate"]*100, color="#2563EB")
for i, (n, r) in enumerate(zip(cohorts["clients"], cohorts["churn_rate"])):
    ax.text(i, r*100 + 0.6, f"{r*100:.0f}%\n(n={n})", ha="center", fontsize=8)
ax.set_title("Churn by onboarding cohort — newer cohorts churn more. But is that a WHY?",
             loc="left", fontweight="bold")
ax.set_ylabel("churn %"); plt.tight_layout(); plt.show()

**Careful.** Newer cohorts churn more — but newer cohorts are also simply *younger relationships*. Is 2025's higher churn a worse vintage, or just less tenure? Cohort and tenure are tangled in a snapshot like this. The honest cut: look at churn **within tenure bands**, across cohorts. (Exercise 1 makes you do it.) The general lesson arrives early: *the first pattern you see is usually two patterns wearing one coat.*

### ✏️ Exercise 1
Create tenure bands with `pd.cut(cli["tenure_months"], [0,24,48,84,144])`, then pivot churn by cohort × band, with a count pivot beside it. Within the same band, do newer cohorts still look worse — or does the vintage story mostly dissolve into a tenure story?

In [ ]:
# your code here


---
## 2. Driver hunting: disciplined group comparisons

The workhorse of "why": split by a candidate driver, compare rates, **always with counts.** The cleanest driver in this book:

In [ ]:
drv = cli.groupby("products_held").agg(n=("client_id","count"), churn=("churned","mean"))
print(drv.round(3))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(drv.index, drv["churn"]*100, marker="o", color="#0D9488", lw=2)
ax.set_title("Churn falls steadily with products held: 34% at one product -> 10% at six",
             loc="left", fontweight="bold")
ax.set_xlabel("products held"); ax.set_ylabel("churn %"); plt.tight_layout(); plt.show()

A clean monotonic gradient — every step down as products rise. This is strong *diagnostic* evidence that product depth and retention travel together. Hold that thought (and your excitement) until section 5, where we ask what it does — and does not — license you to conclude.

---
## 3. The mix trap: adjust before you blame

The RM league table — the chart someone will absolutely put in a deck:

In [ ]:
rm = cli.groupby("relationship_manager").agg(
    n=("client_id","count"), churn=("churned","mean")).sort_values("churn")
print("Best 3:\n", rm.head(3).round(2))
print("\nWorst 3:\n", rm.tail(3).round(2))
print(f"\nSpread: {rm['churn'].min():.0%} to {rm['churn'].max():.0%} — someone is getting fired?")

In [ ]:
# NOT so fast. RMs serve different CLIENT MIXES, and mix drives churn by itself.
# Standardisation: what churn WOULD each RM have, if their only difference were their mix?
seg_rates = cli.groupby("segment")["churned"].mean()          # baseline churn per segment

mix = cli.pivot_table(index="relationship_manager", columns="segment",
                      values="client_id", aggfunc="count").fillna(0)
mix = mix.div(mix.sum(axis=1), axis=0)                        # each RM's segment mix

rm["expected_churn"] = (mix * seg_rates).sum(axis=1)          # mix-implied churn
rm["residual"] = rm["churn"] - rm["expected_churn"]           # what mix CANNOT explain

print(rm.sort_values("residual").round(3).head(3))
print(rm.sort_values("residual").round(3).tail(3))
print(f"\nMix explains only a sliver here: expected churn barely varies "
      f"({rm['expected_churn'].min():.0%}-{rm['expected_churn'].max():.0%}) "
      f"while actual spans {rm['churn'].min():.0%}-{rm['churn'].max():.0%}.")

Two honest findings, and the second matters more:

1. **The method**: never compare raw rates across units with different mixes — compute the mix-*expected* rate first, and judge the **residual**. (Same logic as school boards adjusting results for intake.)
2. **The result**: here, mix explains little — the RM spread survives adjustment. That does **not** convict the worst RMs. It means the *measured* confounder is innocent, and the honest next sentence is: *"the gap isn't mix; candidate explanations are skill, unmeasured client differences, or small-sample noise (n≈40 per RM) — here is how we'd test each."* Diagnostics that ends in a ranked list of suspects **plus a test plan** is finished; diagnostics that ends in a firing is malpractice.

### ✏️ Exercise 2
Do the same standardisation using **city** instead of segment as the mix variable. Does any RM's residual change materially? What does it mean if the answer is "barely"?

In [ ]:
# your code here


---
## 4. The hidden-populations trap

Does having an SIP (a monthly investment plan) retain clients? The overall answer:

In [ ]:
print(cli.groupby("sip_active")["churned"].mean().round(3))
print("\nVerdict from the average: SIP makes ~no difference (20.9% vs 20.0%). Case closed?")

In [ ]:
# Split by segment and the 'no effect' shatters into OPPOSITE effects:
piv_r = cli.pivot_table(index="segment", columns="sip_active", values="churned", aggfunc="mean")
piv_n = cli.pivot_table(index="segment", columns="sip_active", values="churned", aggfunc="count")
piv_r["effect"] = piv_r[1] - piv_r[0]
print((piv_r*100).round(1), "\n\ncounts:\n", piv_n)

In Affluent, HNI and Ultra-HNI, SIP clients churn **less** (in HNI, 6.5 points less). In Mass, SIP clients churn slightly **more**. Averaged together, the opposite effects cancel into a shrug.

> **The average was not wrong. It was answering a question nobody should ask** — "what is the SIP effect for a client who is a blend of all segments?" No such client exists.

This is the everyday cousin of **Simpson's paradox** — the extreme case where an effect doesn't just vanish in aggregate but *reverses sign*. A constructed two-RM example (labelled as constructed, since your data shows the vanishing form rather than full reversal):

In [ ]:
# CONSTRUCTED illustration - classic Simpson's reversal
demo = pd.DataFrame({
    "RM":      ["A","A","B","B"],
    "segment": ["Mass","HNI","Mass","HNI"],
    "clients": [ 90,   10,   10,   90 ],
    "churned": [ 27,    1,    4,   14 ],
})
demo["rate"] = demo["churned"] / demo["clients"]
print(demo)

overall = demo.groupby("RM").apply(lambda x: x["churned"].sum()/x["clients"].sum(), include_groups=False)
print("\nOverall churn -> RM A:", f"{overall['A']:.0%}", "| RM B:", f"{overall['B']:.0%}", "  (B looks WORSE)")
print("Within Mass  -> A: 30% vs B: 40%   (A better)")
print("Within HNI   -> A: 10% vs B: 15.6% (A better)")
print("\nA is better in EVERY segment, yet worse overall - because A holds the high-churn Mass book.")
print("Aggregation direction can REVERSE a conclusion. Always ask: better AT WHAT MIX?")

### ✏️ Exercise 3
Run the SIP-by-segment analysis for `risk_profile` instead. Overall, Aggressive clients look *safest* (16% churn). Does that survive the within-segment cut — and what does the HNI row tell you about who chooses "Aggressive"?

In [ ]:
# your code here


---
## 5. What "why" can honestly claim

You now hold a gradient (products vs churn), an adjusted league table, and a segment-split SIP effect. Time for the module's most important paragraph.

**Diagnostic analytics establishes association, rules out measured confounders, and ranks hypotheses. It does not, by itself, establish causation.** The products gradient is consistent with at least three worlds:

- **A → B**: holding more products *causes* loyalty (switching costs, engagement)
- **B → A**: loyal clients *acquire* more products over their longer relationships
- **C → A and B**: satisfaction (unmeasured) drives both buying and staying

Your snapshot cannot separate these — and a recommendation ("cross-sell to reduce churn!") is only justified under world one. The honest deliverable names all three and states what would distinguish them: timing data (did products precede retention?), or best of all an **experiment** — offer a product push to a random half of at-risk clients and compare. Where experiments are impossible, quasi-experimental designs exist (that's advanced-course territory); what is *always* possible is refusing to dress an association up as a mechanism.

**The diagnostic deliverable, summarised:** the decomposed variance (5A), the adjusted comparison, the segment-split effect, three candidate mechanisms, and the test that would separate them. That final section is what distinguishes an analyst from a chart-maker.

---
*AI disclosure: ______*

In [ ]:
# workspace
